In [ ]:
import pandas as pd
from collections import Counter
import re

def analyze_keyword_frequencies(file_path, min_count=10):
    # 1. 데이터 로드
    df = pd.read_csv(file_path)
    
    # ---------------------------------------------------------
    # [중복 제거] 논문 ID 기준으로 중복 행 제거
    df_unique = df.drop_duplicates(subset=['source_id']).copy()
    print(f"✅ 중복 제거 완료: {len(df)}개 행 -> {len(df_unique)}개 논문으로 압축됨")
    # ---------------------------------------------------------
    
    # 2. 키워드 분리 및 정제
    all_keywords = []
    for k_str in df_unique['keywords'].dropna():
        # 구분자( , 또는 ; ) 처리 및 공백 제거, 소문자화
        words = [w.strip().lower() for w in str(k_str).replace(';', ',').split(',') if w.strip()]
        all_keywords.extend(words)

    # 3. 빈도 계산
    counts = Counter(all_keywords)
    
    # 4. 전체 데이터를 데이터프레임으로 변환
    # most_common()에 인자를 주지 않으면 전체 키워드를 가져옵니다.
    full_freq_df = pd.DataFrame(counts.most_common(), columns=['Keyword', 'Count'])
    
    # ---------------------------------------------------------
    # [핵심 수정] 10회 이상 등장한 키워드만 필터링
    freq_df = full_freq_df[full_freq_df['Count'] >= min_count].copy()
    # ---------------------------------------------------------
    
    # 5. 언어 판별 함수 적용
    def get_lang_type(text):
        if re.search('[가-힣]', str(text)):
            return '한글/혼용'
        return '영문'

    freq_df['Type'] = freq_df['Keyword'].apply(get_lang_type)
    
    # 6. 결과 출력 및 저장
    print(f"\n📊 {min_count}회 이상 등장한 키워드 분석 현황")
    print("-" * 60)
    if not freq_df.empty:
        print(freq_df)
        # 파일로 저장 (분석용)
        output_name = f'키워드_빈도_분석_{min_count}회이상.csv'
        freq_df.to_csv(output_name, index=False, encoding='utf-8-sig')
        print("-" * 60)
        print(f"✅ 필터링된 키워드 총 {len(freq_df)}개가 '{output_name}'에 저장되었습니다.")
    else:
        print(f"❌ {min_count}회 이상 등장한 키워드가 없습니다.")
    
    return freq_df

if __name__ == "__main__":
    file_name = '법학_AI_상세_및_인용데이터.csv'
    # 10회 이상 등장하는 키워드만 분석
    freq_result = analyze_keyword_frequencies(file_name, min_count=10)

In [ ]:
freq_result.to_csv("키워드_빈도_상위_100.csv")

In [ ]:
import pandas as pd
from collections import Counter
import re
import os

def analyze_keywords_with_mapping(data_file, mapping_file, top_n=50):
    if not os.path.exists(mapping_file):
        print(f"❌ 매핑 파일을 찾을 수 없습니다: {mapping_file}")
        return
    
    # 1. 매핑 사전 생성
    df_map = pd.read_csv(mapping_file)
    df_map = df_map.dropna(subset=['Keyword', 'Target'])
    
    # 영문/한글 구분 없이 모든 키워드를 소문자 처리하여 사전에 등록
    # (Key: 원본 키워드, Value: 변환될 Target 키워드)
    mapping_dict = dict(zip(
        df_map['Keyword'].astype(str).str.lower(), 
        df_map['Target'].astype(str)
    ))
    
    # 2. 메인 데이터 로드 및 중복 제거
    df = pd.read_csv(data_file)
    # 인용 정보로 인해 늘어난 행을 논문ID 기준으로 압축 (가장 중요!)
    df_unique = df.drop_duplicates(subset=['source_id']).copy()
    print(f"✅ 데이터 로드 완료: {len(df_unique)}개 논문 분석")

    # 3. 키워드 치환 및 수집
    final_keywords = []
    for k_str in df_unique['keywords'].dropna():
        # 분리 및 정제
        raw_words = [w.strip().lower() for w in str(k_str).replace(';', ',').split(',') if w.strip()]
        
        for word in raw_words:
            # 매핑 사전에 있으면 Target으로 변환, 없으면 원본 유지
            mapped_word = mapping_dict.get(word, word)
            final_keywords.append(mapped_word)

    # 4. 빈도 계산
    counts = Counter(final_keywords)

    all_counts = counts.most_common() 
    
    # 5. 데이터프레임 변환
    freq_df = pd.DataFrame(all_counts, columns=['Keyword', 'Count'])
    
    # 한글/영문 타입 판별 (시각화용)
    def get_lang_type(text):
        if re.search('[가-힣]', text):
            return '한글/혼용'
        return '영문'
    freq_df['Type'] = freq_df['Keyword'].apply(get_lang_type)

    # 6. 출력
    print(f"\n📊 매핑 적용 후 상위 {top_n}개 키워드")
    print("-" * 60)
    print(freq_df.head(20)) # 상위 20개만 우선 출력
    print("-" * 60)
    
    return freq_df

if __name__ == "__main__":
    DATA_FILE = '법학_AI_상세_및_인용데이터.csv'
    MAPPING_FILE = '키워드 변환 표.csv' # 사용자의 매핑 파일명에 맞게 수정하세요
    
    result = analyze_keywords_with_mapping(DATA_FILE, MAPPING_FILE)
    
    # 결과 저장 (필요 시)
result.to_csv('매핑후_키워드_빈도.csv', index=False, encoding='utf-8-sig')

### 키워드 네트워크 분석

In [26]:
import pandas as pd
import networkx as nx
import community.community_louvain as community_louvain
from itertools import combinations
from collections import Counter
import os
import re

def perform_refined_korean_analysis(data_file, mapping_file, min_edge_weight=2):
    # 1. 제외 키워드 (이미 매핑되어 '인공지능'이 된 것들도 필터링함)
    EXCLUDE_KEYWORDS = ['인공지능', 'ai', 'artificial intelligence']
    
    # 2. 매핑 사전 로드 및 키 정제
    mapping_dict = {}
    if os.path.exists(mapping_file):
        df_map = pd.read_csv(mapping_file).dropna(subset=['Keyword', 'Target'])
        # 사전의 Keyword도 공백 제거 및 소문자화하여 저장
        for _, row in df_map.iterrows():
            key = str(row['Keyword']).strip().lower()
            mapping_dict[key] = str(row['Target']).strip()

    # 3. 데이터 로드 및 중복 제거
    df = pd.read_csv(data_file)
    df_unique = df.drop_duplicates(subset=['source_id']).copy()
    
    processed_docs = []
    ko_pattern = re.compile('[가-힣]')

    for k_str in df_unique['keywords'].dropna():
        # 기본 분리
        raw_words = [w.strip().lower() for w in str(k_str).replace(';', ',').split(',') if w.strip()]
        
        refined_words = []
        for w in raw_words:
            # --- [강화된 정제 로직 시작] ---
            # 1. 괄호와 그 안의 내용 삭제 (예: '인공지능(ai)' -> '인공지능')
            clean_w = re.sub(r'\(.*\)', '', w).strip()
            
            # 2. 매핑 사전 적용 (괄호 제거 전/후 모두 체크)
            # 먼저 원본(w)으로 찾아보고, 없으면 정제본(clean_w)으로 사전 검색
            target_w = mapping_dict.get(w, mapping_dict.get(clean_w, clean_w))
            # --- [강화된 정제 로직 종료] ---
            
            if pd.isna(target_w) or str(target_w).lower() == 'nan' or target_w == '':
                continue
            
            # 최종 필터링: 제외어에 없고 + 한글이 포함된 경우만
            if target_w.lower() not in EXCLUDE_KEYWORDS:
                if ko_pattern.search(target_w):
                    refined_words.append(target_w)
        
        refined_words = list(set(refined_words))
        if len(refined_words) >= 2:
            processed_docs.append(refined_words)

    # 4. 네트워크 생성 및 분석 (이하 로직 동일)
    G = nx.Graph()
    edge_counts = Counter()
    for words in processed_docs:
        for pair in combinations(sorted(words), 2):
            edge_counts[pair] += 1

    for (node1, node2), weight in edge_counts.items():
        if weight >= min_edge_weight:
            G.add_edge(node1, node2, weight=weight)

    if G.number_of_nodes() == 0:
        print("⚠️ 분석할 데이터가 없습니다. 필터를 확인하세요.")
        return

    partition = community_louvain.best_partition(G, weight='weight', random_state=42)
    
    # 군집 결과 출력
    clusters = {}
    for node, cluster_id in partition.items():
        clusters.setdefault(cluster_id, []).append(node)

    print(f"🚀 분석 완료: 총 {len(clusters)}개 군집 발견")
    for cluster_id, nodes in sorted(clusters.items()):
        top_keywords = sorted(nodes, key=lambda x: G.degree(x, weight='weight'), reverse=True)[:7]
        print(f"군집 {cluster_id+1}: {', '.join(top_keywords)}")

    return G, partition

if __name__ == "__main__":
    perform_refined_korean_analysis('법학_AI_상세_및_인용데이터.csv', '키워드 변환 표.csv')

🚀 분석 완료: 총 21개 군집 발견
군집 1: 법인격, 제조물책임, 로봇, 자율주행자동차, 위험책임, 자율성, 전자인
군집 2: 인공지능의 범죄능력, 인공지능의 형사책임능력
군집 3: 데이터구조, 데이터셋, 물건발명, 인공지능발명
군집 4: 옵트아웃, ai 학습데이터, 메타데이터
군집 5: 딥러닝, 머신러닝, 기계학습, 심층학습, 진보성, 발명, ai발명
군집 6: 인공지능규범, 인공지능백서
군집 7: 투명성, 편향, 차별, 공정성, 편향성, 설명가능성, 책임성
군집 8: 기술주권, 소버린 ai
군집 9: 직업교육, 직업능력개발
군집 10: 의사표시의 주체, 전자문서 및 전자거래기본법, 컴퓨터시스템 운영자
군집 11: 인공지능 법, 인공지능법, 인공지능기본법, 인공지능 거버넌스, 인공지능 규제, 고위험 인공지능, 개인정보보호법
군집 12: 법률서비스, 변호사법, 사법접근권
군집 13: 강인공지능, 약인공지능
군집 14: 생성형 인공지능, 저작권, 공정이용, 빅데이터, 저작자, 개인정보보호, 저작물
군집 15: 조정, 중재, 협상
군집 16: 알고리즘, 규제, 묵시적 담합, 헌법, 동조적 행위, 디지털 카르텔, 기본권
군집 17: 양형, 재범예측
군집 18: 인공지능 저작물, 독창성, 중국 저작권법, 인공지능 저작물의 저작자, 인공지능 저작자
군집 19: 개인정보, 4차 산업혁명, 인공지능 로봇, 사물인터넷, 지능정보사회, 특허권, 지식재산권
군집 20: 법경제학, 자동화
군집 21: ai범죄, 배후정범


In [27]:
import pandas as pd
import networkx as nx
import community.community_louvain as community_louvain
from itertools import combinations
from collections import Counter
import os
import re

def perform_and_save_clusters(data_file, mapping_file, min_edge_weight=2):
    # 1. 설정 및 로드
    EXCLUDE_KEYWORDS = ['인공지능', 'ai', 'artificial intelligence']
    mapping_dict = {}
    if os.path.exists(mapping_file):
        df_map = pd.read_csv(mapping_file).dropna(subset=['Keyword', 'Target'])
        mapping_dict = dict(zip(df_map['Keyword'].astype(str).str.lower(), df_map['Target'].astype(str)))

    # 2. 데이터 중복 제거
    df = pd.read_csv(data_file)
    df_unique = df.drop_duplicates(subset=['source_id']).copy()
    
    # 3. 키워드 정제 및 한국어 필터링
    processed_docs = []
    ko_pattern = re.compile('[가-힣]')

    for k_str in df_unique['keywords'].dropna():
        raw_words = [w.strip().lower() for w in str(k_str).replace(';', ',').split(',') if w.strip()]
        refined_words = []
        for w in raw_words:
            clean_w = re.sub(r'\(.*\)', '', w).strip()
            target_w = mapping_dict.get(w, mapping_dict.get(clean_w, clean_w))
            
            if target_w.lower() not in EXCLUDE_KEYWORDS and ko_pattern.search(target_w):
                refined_words.append(target_w)
        
        refined_words = list(set(refined_words))
        if len(refined_words) >= 2:
            processed_docs.append(refined_words)

    # 4. 네트워크 및 군집화
    G = nx.Graph()
    edge_counts = Counter()
    for words in processed_docs:
        for pair in combinations(sorted(words), 2):
            edge_counts[pair] += 1

    for (node1, node2), weight in edge_counts.items():
        if weight >= min_edge_weight:
            G.add_edge(node1, node2, weight=weight)

    # Louvain 군집화
    partition = community_louvain.best_partition(G, weight='weight', random_state=42)

    # ---------------------------------------------------------
    # 5. [핵심] 결과 데이터프레임 생성 및 저장
    # ---------------------------------------------------------
    cluster_results = []
    for node, cluster_id in partition.items():
        # 각 노드(키워드)의 가중치 기반 연결 정도(Weighted Degree) 계산
        degree = G.degree(node, weight='weight')
        cluster_results.append({
            'Keyword': node,
            'Cluster_ID': cluster_id + 1,  # 1번부터 시작하도록
            'Importance': degree           # 군집 내 중요도 지표
        })

    # 데이터프레임 변환 및 정렬 (군집별, 중요도순)
    result_df = pd.DataFrame(cluster_results)
    result_df = result_df.sort_values(by=['Cluster_ID', 'Importance'], ascending=[True, False])

    # CSV 저장
    output_filename = '키워드_군집_매핑_결과.csv'
    result_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    # 6. 요약 출력
    print(f"✅ 분석 및 저장 완료!")
    print(f"📂 결과 파일: {output_filename}")
    print("-" * 50)
    for cid in sorted(result_df['Cluster_ID'].unique()):
        top_5 = result_df[result_df['Cluster_ID'] == cid].head(5)['Keyword'].tolist()
        print(f"군집 {cid}: {', '.join(top_5)} ...")
    
    return result_df

if __name__ == "__main__":
    df_result = perform_and_save_clusters('법학_AI_상세_및_인용데이터.csv', 'keyword_mapping.csv')

✅ 분석 및 저장 완료!
📂 결과 파일: 키워드_군집_매핑_결과.csv
--------------------------------------------------
군집 1: 저작권, 생성형 인공지능, 공정이용, 빅데이터, 생성형 ai ...
군집 2: 조정, 중재, 협상 ...
군집 3: 데이터구조, 데이터셋, 물건발명, 인공지능발명 ...
군집 4: 인공지능 저작물, 독창성, 중국 저작권법, 인공지능 저작물의 저작자, 인공지능 저작자 ...
군집 5: 투명성, 차별, 공정성, 개인정보보호, 편향성 ...
군집 6: 진보성, ai발명, 발명, 발명자, 소프트웨어 ...
군집 7: 양형, 재범예측 ...
군집 8: 개인정보, 사물인터넷, 4차 산업혁명, 지능정보사회, 특허권 ...
군집 9: 인공지능기본법, 인공지능 거버넌스, 고위험 인공지능, 인공지능 기본법, 고영향 인공지능 ...
군집 10: 법인격, 제조물책임, 로봇, 위험책임, 전자인 ...
군집 11: 다크패턴, 소비자보호 ...
군집 12: 법경제학, 자동화 ...
군집 13: ai범죄, 배후정범 ...
군집 14: 알고리즘, 묵시적 담합, 헌법, 동조적 행위, 디지털 카르텔 ...
군집 15: 의료 인공지능, 설명의무, 의료행위, 의료기기, 법적 인격 ...
군집 16: 인공지능의 범죄능력, 인공지능의 형사책임능력 ...
군집 17: 옵트아웃, ai 학습데이터, 메타데이터 ...
군집 18: 로봇윤리, 인공적 도덕행위자 ...
군집 19: 인공지능규범, 인공지능백서 ...
군집 20: 기술주권, 소버린 ai ...
군집 21: 직업교육, 직업능력개발 ...
군집 22: 의사표시의 주체, 전자문서 및 전자거래기본법, 컴퓨터시스템 운영자 ...
군집 23: 법률서비스, 변호사법, 사법접근권 ...
군집 24: 강인공지능, 약인공지능 ...


In [ ]:
import pandas as pd
import re

def find_english_only_keyword_papers(file_path):
    # 1. 데이터 로드 및 중복 제거
    df = pd.read_csv(file_path)
    df_unique = df.drop_duplicates(subset=['source_id']).copy()
    
    # 2. 한글 패턴 정의
    ko_pattern = re.compile('[가-힣]')
    
    # 3. 영문 키워드만 가진 논문 필터링
    # 키워드가 비어있지 않으면서(notna), 한글이 전혀 없는(not search) 논문 선택
    english_only_mask = df_unique['keywords'].apply(
        lambda x: pd.notna(x) and not bool(ko_pattern.search(str(x)))
    )
    
    df_english_only = df_unique[english_only_mask]
    
    # 4. 결과 출력
    print(f"📊 분석 결과")
    print("-" * 60)
    print(f"✅ 전체 논문 수: {len(df_unique)}건")
    print(f"✅ 영문 키워드만 있는 논문 수: {len(df_english_only)}건")
    print("-" * 60)
    
    if not df_english_only.empty:
        # 주요 정보만 상위 10개 출력 (제목, 키워드)
        print("📑 [상위 10개 목록]")
        print(df_english_only[['raw_citation', 'keywords']].head(10))
        
        # 결과 저장
        df_english_only.to_csv('영문키워드_전용_논문목록.csv', index=False, encoding='utf-8-sig')
        print(f"\n📂 상세 목록이 '영문키워드_전용_논문목록.csv'로 저장되었습니다.")
    else:
        print("🔍 모든 논문에 최소 하나 이상의 한글 키워드가 포함되어 있습니다.")

    return df_english_only

if __name__ == "__main__":
    file_name = '법학_AI_상세_및_인용데이터.csv'
    eng_papers = find_english_only_keyword_papers(file_name)